# LociSimiles Contextual Latin BERT Example

Run contextual token-level retrieval on the sample Latin corpus.

This notebook prefers a local LatinBERT directory at `./models/latinbert` and falls back to the HuggingFace model `ashleygong03/bamman-burns-latin-bert`.

In [1]:
from pathlib import Path
from locisimiles.document import Document
from locisimiles.evaluator import IntertextEvaluator
from locisimiles.pipeline import LatinBertRetrievalPipeline, pretty_print

/Users/julianschelb/.pyenv/versions/locisimiles-test/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
base = Path('.')
query_doc = Document(base / 'hieronymus_samples.csv', author='Hieronymus')
source_doc = Document(base / 'vergil_samples.csv', author='Vergil')

local_latinbert_path = base / 'models' / 'latinbert'
latinbert_model_name = 'ashleygong03/bamman-burns-latin-bert'

pipeline_kwargs = {
    'top_k': 10,
    'similarity_threshold': 0.85,
    'max_length': 256,
    'min_token_length': 2,
    'use_stopword_filter': True,
}
if local_latinbert_path.exists():
    pipeline_kwargs['model_path'] = local_latinbert_path
    print(f'Using local LatinBERT model path: {local_latinbert_path}')
else:
    pipeline_kwargs['model_name'] = latinbert_model_name
    print(f'Using HuggingFace LatinBERT model: {latinbert_model_name}')

pipeline_contextual = LatinBertRetrievalPipeline(**pipeline_kwargs)

Using HuggingFace LatinBERT model: dbamman/latin-bert


OSError: dbamman/latin-bert is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`

In [ ]:
results_contextual = pipeline_contextual.run(query=query_doc, source=source_doc, top_k=10)
pretty_print(results_contextual)

In [ ]:
evaluator = IntertextEvaluator(
    query_doc=query_doc,
    source_doc=source_doc,
    ground_truth_csv=base / 'ground_truth.csv',
    pipeline=pipeline_contextual,
    top_k=10,
    threshold=0.85,
)
print(evaluator.evaluate(average='macro', with_match_only=True))

In [ ]:
pipeline_contextual.to_csv(base / 'results_contextual_bert.csv', results=results_contextual)
pipeline_contextual.to_json(base / 'results_contextual_bert.json', results=results_contextual)